# Øvelser: tekst som data

**Social Data Science 1: lektion 7**

Øvelserne bruger det datasæt, vi har preprocesseret sammen, `ft_lovforslag.csv`: 3.070 debatter
fra 1. behandlingen af lovforslag i Folketinget, oktober 2007 til november 2022.

**Del A** er simple, deskriptive **bag-of-word**s-øvelser: tæl ord, find de hyppigste, se på
fordelingen. **Del B** bygger videre på det, I så i slides, men som mere "rigtige"
undersøgelser.

Under **Hjælp** står, hvor I har set de funktioner, I skal bruge:

- **Slides**: lektion 7-slides, med navnet på slidet.
- **Preprocess-notebooken**: `l7_preprocess_dkparl_trinvis`, med trinnets nummer.
- **spaCy-notebooken**: `l7_spacy_supplement`, med trinnets nummer.


Kodecellerne er delvist skrevet for jer. I skal udfylde resten:

- `____` er en plads, I skal udfylde.

Kører I en celle, hvor der stadig står `____`, får I en fejl som `NameError: name '____' is not defined`.
Det betyder bare, at der mangler noget.

Det er forsøgt at gøre hjælpen gradvist mindre i løbet af notebooken.

---

## Opsætning

Kør cellerne. De indlæser data og definerer stopordslisten fra slides (*Fra tekst til tabel*).
Ret stien, så den peger på jeres egen kopi af `ft_lovforslag.csv`.

In [ ]:
import pandas as pd
import numpy as np
from plotnine import *


sti = "/Users/jeppefl/Library/CloudStorage/OneDrive-AalborgUniversitet/01_work/01_undervisning/02_sds1/01_slides/lektion-7/resources/ft_lovforslag.csv"
korpus = pd.read_csv(sti)

print(korpus.shape)
print(korpus.columns.tolist())

In [ ]:
stopord = set("""og i jeg det at en den til er som på de med han af for ikke
der var mig sig men et har om vi min havde ham hun nu over da fra du ud sin dem
os op man hans hvor eller hvad skal selv her alle vil blev kunne ind når være dog
jo dette dig deres end mit også under have dens hvis dine disse hvem vores jer
sådan andre nogle bliver blive kan så ved mod meget rigtig tak hr fru ordfører
ordføreren forslag lovforslag lovforslaget minister ministeren værsgo altså godt
synes mener bare lige hele netop sige siger""".split())

print(len(stopord), "stopord")

Hele materialet er cirka 22 millioner ord. Tabellen med ét ord per række, som I laver i A2 og
A4, fylder derfor en del i hukommelsen. Har jeres computer under 8 GB RAM, så arbejd på et
udsnit ved at køre denne linje efter indlæsningen:

```python
korpus = korpus.sample(1000, random_state=1)
```

---

# Del A: Bag-of-words, deskriptivt

Her gør vi det, I kender fra lektion 6, hvor vi deler teksterne op i ord og tæller. Målet er at få en
fornemmelse for materialet, før vi analyserer det.

---

## A1. Overblik over korpusset

Før vi tæller ord, skal vi vide, hvad vi har, herunder hvor mange debatter er der fra hvert
politikområde og hvert år?

**Opgave:**

1. Tæl antallet af debatter per politikområde (`omraade`).
2. Tæl antallet af debatter per år (`aar`), sorteret efter år.
3. Lav en krydstabel med år som rækker og politikområde som kolonner.

**Hjælp:**

- Preprocess-notebooken, trin 9 og 10: `.value_counts()` på `omraade` og `.value_counts().sort_index()` på `aar`.
- Slides, *Validering først*: `pd.crosstab()` med to kolonner.

In [ ]:
# Din kode her

# Debatter per politikområde
print(korpus["____"].value_counts())
print()

# Debatter per år, sorteret efter år (ikke efter antal)
print(korpus["aar"].value_counts().____())

In [ ]:
# Krydstabel med år som rækker, politikområde som kolonner
omraade_aar = pd.crosstab(____, ____)

print(omraade_aar)

**Kig efter i viewer:** Åbn krydstabellen. Hvilke celler er små? Find kolonnen `Udlændinge`.

**Til diskussion:**

- Hvorfor er der færre debatter i 2007 og 2022?
- Hvor mange debatter om Udlændinge er der typisk per år? Hvad betyder det, hvis vi vil sige
  noget om udviklingen over tid inden for ét område?

*Jeres svar:*

---

## A2. Fra tekst til ord

Brug opskriften fra slides til at lave teksterne om til én lang Series med ét ord per række.
Indekset husker, hvilken debat ordet kom fra.

**Opgave:**

1. Lav en Series `ord`: små bogstaver, erstat alt, der ikke er bogstaver, tal eller mellemrum,
   med mellemrum, del op, og fold ud.
2. Hvor mange **tokens** og hvor mange **typer** er der?
3. Tæl antallet af ord i hver debat, og gem det som en ny kolonne `n_ord` i `korpus`.
4. Beskriv fordelingen af `n_ord`, og lav en oversigt med antal debatter og medianen af `n_ord`
   for hvert politikområde.

**Hjælp:**

- Slides, *Hvordan den scorer*: hele opskriften fra `korpus["tekst"]` til `.explode()`.
- Slides, *Kør den på korpusset*: `.groupby(level=0)` grupperer efter debatten.
- Preprocess-notebooken, trin 5.2: `oversigt` med `.agg(antal="size", median_tegn="median")`.

In [ ]:
# Din kode her

# Fra tekst til ét ord per række
ord = (korpus["tekst"].astype(str)
       .str.____()                                    # små bogstaver
       .str.replace(r"[^\wæøå ]", " ", regex=True)   # alt andet end bogstaver, tal og mellemrum bliver til mellemrum/[space]
       .str.____()                                    # del op i ord
       .____())                                       # fold ud med ét ord per række

# Tokens er alle ord. Typer er forskellige ord.
print("tokens:", ____, "| typer:", ____)

In [ ]:
# Antal ord i hver debat (indekset er debatten)
korpus["n_ord"] = ord.groupby(level=0).____()

print(korpus["n_ord"].sum() == len(ord))   # tjek om det giver `True`

# Fordelingen
print(korpus["n_ord"].describe().round(0))

In [ ]:
# Antal debatter og median af n_ord per område
oversigt = (korpus
            .groupby("____")["____"]
            .agg(antal="size", median_ord="____")
            .sort_values("median_ord", ascending=False))

print(oversigt)

**Kig efter i viewer:** Sortér `korpus` efter `n_ord`. Hvilke debatter er de korteste og de længste? Hvad handler de om?

**Til diskussion:**

- Hvor meget varierer antallet af ord fra debat til debat?
- Hvilke politikområder har de længste debatter? Er antal ord et godt mål for, hvor meget en
  debat fylder politisk?
- Slides målte temaer *per 1.000 ord*. Hvorfor er det nødvendigt her?

*Jeres svar:*

---

## A3. De hyppigste ord, og hvad stopordene fjerner

Hvilke ord bruges mest i Folketinget? Først uden at fjerne noget, derefter uden stopord. Vi
fører regnskab med tokens og typer, som i spaCy-notebooken.

**Opgave:**

1. Find de 20 hyppigste ord i `ord`.
2. Lav tabellen `kaede` og funktionen `tael()` fra spaCy-notebooken, og tæl `ord`.
3. Lav en tabel `fjernet_w` med de fjernede stopord og deres antal. Fjern stopordene, gem
   resultatet som `indhold`, og tæl igen.
4. Find de 25 hyppigste ord i `indhold`.

**Hjælp:**

- spaCy-notebooken, trin 4: `kaede` og `tael()`.
- spaCy-notebooken, trin 5.4: `fjernet_w` med `.value_counts().reset_index()`, og filteret `~ ... .isin(stopord)`.
- Slides, *Hvad fylder i scoren?*: `.value_counts().head()`.

In [ ]:
# Din kode her

# De 20 hyppigste ord
print(ord.____().head(____))

In [ ]:
# Regnskab over tokens og typer
kaede = pd.DataFrame(columns=["trin", "tokens", "typer"])

def tael(trin, former):
    kaede.loc[len(kaede)] = [trin, ____, ____]   # antal tokens og antal typer i former
    print(kaede)

tael("Rå tokens", ord)

In [ ]:
# De stopord, der bliver fjernet, med deres antal
fjernet_w = ord[ord.isin(____)].value_counts().reset_index()

# Behold kun de ord, der IKKE er stopord (~ betyder "ikke")
indhold = ord[____]

tael("+ fjern stopord", indhold)

In [ ]:
# De 25 hyppigste ord i indhold
# ... [SKRIV KODE HER]

**Kig efter i viewer:** Åbn `fjernet_w`. Hvor langt nede på listen skal I, før I finder et ord, I ville have beholdt?

**Til diskussion:**

- Hvilke ord dominerer, før stopordene er fjernet? Står der noget, der siger noget om politik?
- Fjerner stopordene flest tokens eller flest typer? Er mønsteret det samme som i spaCy-notebooken?
- Hvilke ord i top 25 efter stopordene overrasker jer? Kan I forklare, hvorfor de er der?

*Jeres svar:*

---

## A4. De hyppigste ord per politikområde

Kan man se forskel på politikområderne ud fra de hyppigste ord? For at koble ordene til
områderne laver vi tabellen `lang` fra slides med ét ord per række og år og område ved siden af.

**Opgave:**

1. Lav tabellen `lang` ud fra `ord` og kobl `aar` og `omraade` på fra `korpus`.
2. Lav `indhold` om, så den er `lang` uden stopord.
3. Find de 10 hyppigste ord i hvert politikområde, og vis dem som én liste per område.

**Hjælp:**

- Slides, *Tre temaordbøger*: `ord.rename("ord").to_frame().join(korpus[["aar", "omraade"]])`.
- spaCy-notebooken, trin 5.6: `fjernet_pos` med `.groupby(...)[...].value_counts().groupby(level=0).head(10)`.
- Slides, *Hvad handler klyngerne om?*: `.groupby(...)["ord"].agg(list)`.

In [ ]:
# Din kode her
# Én række per ord, med år og område koblet på via indekset
lang = ord.rename("ord").to_frame().join(korpus[["____", "____"]])

print(len(lang) == len(ord))   # tjek om det giver `True`

# lang uden stopord. Nu er ordene en kolonne i en tabel, ikke en Series
indhold = lang[~lang["____"].isin(stopord)]

print(indhold.head())

In [ ]:
# Tæl ordene inden for hvert område, og behold de 10 hyppigste i hvert
top10 = (indhold
         .groupby("____")["____"]
         .value_counts()
         .groupby(level=0)
         .head(____)
         .reset_index())

# Én liste med ord per område
print(top10.groupby("omraade")["ord"].agg(____))

**Kig efter i viewer:** Åbn `top10` (eller hvad i kalder variablen), og filtrér på ét område ad gangen.

**Til diskussion:**

- Hvor mange af de ti hyppigste ord er ens på tværs af områderne?
- Hvilke ord gør det muligt at genkende et område?
- Hvad fortæller det om hyppighed som mål for, hvad der er *særligt* for et område?

*Jeres svar:*

---

## A5. Få ord bruges meget, mange ord bruges sjældent

Ordfrekvenser er meget skævt fordelt. Det er grunden til, at en document-term matrix er næsten
tom (slides, *Document-term matrix*).

**Opgave:**

1. Hvor mange typer optræder kun én gang i hele korpusset (*hapax*)? Hvor stor en andel af alle
   typer er det?
2. Hvor stor en andel af alle tokens udgør de 10 hyppigste ord?
3. Beregn dokumentfrekvensen: i hvor mange debatter står hvert ord? Hvor mange typer står kun
   i én debat?
4. *(Valgfri)* Tegn ordenes frekvens mod deres rang med logaritmiske akser.

**Hjælp:**

- spaCy-notebooken, trin 5.7: `dokfrekvens` og `(dokfrekvens == 1).sum()`.
- Slides, *Document-term matrix*: `ord.rename("ord").reset_index().drop_duplicates()["ord"].value_counts()`.
- Slides, *Tegn det*: plotnine. Brug `scale_x_log10()` og `scale_y_log10()`.

In [ ]:
# Din kode her
frekvens = ord.value_counts()

# De 10 hyppigste ords andel af alle tokens
print("Top 10's andel af tokens:", round(____ / frekvens.sum(), 3))

In [ ]:
# Dokumentfrekvens (tæl hvert ord højst én gang per debat)
dokfrekvens = ord.rename("ord").reset_index().____()["ord"].value_counts()

print("Typer i kun én debat:", ____, "af", len(dokfrekvens))

In [ ]:
# (Valgfri) Frekvens mod rang
plot = frekvens.reset_index(drop=True).rename("frekvens").to_frame()
plot["rang"] = plot.index + 1      # det hyppigste ord får rang 1

(ggplot(plot, aes(x="____", y="____"))
 + geom_line()
 # ... logaritmisk x-akse
 # ... logaritmisk y-akse
 + labs(x="Rang (log)", y="Frekvens (log)", title="Ordfrekvens mod rang")
 + theme_classic())

**Kig efter i viewer:** Åbn `frekvens`. Rul til bunden. Hvilke slags ord står kun én gang?

**Til diskussion:**

- Hvad betyder fordelingen for en DTM med én kolonne per type?
- Er ord, der kun står i én debat, støj? Giv et eksempel på, hvornår de kunne være det
  interessante.

*Jeres svar:*

---

## A6. Tilbage til teksten: konkordans

Et ord, der tælles, skal også læses i sin sammenhæng. Vi ser på ordet *retssikkerhed*.

**Opgave:**

1. Tæl, hvor mange gange *retssikkerhed* (inklusive bøjninger som *retssikkerheden*) står i hver
   debat. Lav en oversigt per politikområde med antal forekomster og antal debatter, der nævner
   ordet.
2. Hvilket område nævner ordet flest gange? Hvilke enkeltdebatter nævner det mest?
3. Lav en konkordans: find ordet med 60 tegn før og efter, og læs 8 tilfældige eksempler fra det
   område, der nævner ordet mest.

**Hjælp:**

- Slides, *Tilbage til teksten*: `.str.findall(r"(?i).{0,60}\b\w*krise\w*.{0,60}")`, `.explode()`, `.dropna()` og `.sample(8, random_state=1)`.
- Preprocess-notebooken, trin 9: `.str.contains()`. Her skal I bruge `.str.count()`, som tæller i stedet.
- Preprocess-notebooken, trin 5.2: `.agg(...)` med navngivne kolonner.

In [ ]:
# Din kode her

# Antal gange ordet står i hver debat. \w* fanger bøjninger som -en og -ens.
korpus["retssikkerhed"] = korpus["tekst"].str.____(r"(?i)\bretssikkerhed\w*")

oversigt = (korpus
            .groupby("omraade")["retssikkerhed"]
            .agg(forekomster="____",                      # antal forekomster i alt
                 debatter=lambda s: (s > 0).sum())       # antal debatter, der nævner ordet
            .sort_values("forekomster", ascending=False))

print(oversigt)
print()

# De 5 debatter, der nævner ordet flest gange
print(korpus.nlargest(____, "____")[["aar", "omraade", "titel", "retssikkerhed"]])

In [ ]:
# Konkordans (ordet med op til 60 tegn før og efter.)
eksempler = (korpus["tekst"]
             .str.findall(____)
             .explode()
             .dropna()
             .str.replace("\n", " "))

# Kun eksempler fra det område, der nævner ordet mest (første række i oversigt)
oeverst = oversigt.index[0]
fra_omraadet = eksempler[korpus.loc[eksempler.index, "omraade"] == oeverst]

for linje in fra_omraadet.sample(____, random_state=1):
    print("...", linje, "...")

**Kig efter i viewer:** Sortér `korpus` efter `retssikkerhed`. Læs titlerne på de debatter, der nævner ordet mest.

**Til diskussion:**

- Havde I forventet det område? Hvad handler retssikkerhed om, når det bruges der?
- Hvor mange debatter står bag tallet? Hvad betyder det for, hvordan tallet kan tolkes?

*Jeres svar:*

---

# Del B: Udvidelser

Øvelserne bygger videre på det, I så i slides. Hver øvelse tager én metode og gør den mere
"nuanceret".

Øvelserne bruger `ord`, `lang` og `indhold` fra Del A. Kør Del A først.

---

## B1. Nøgleord: hvad er særligt for et område?

A4 viste, at de hyppigste ord er de samme overalt. **Nøgleord** (*keyness*, Hunston) er ord,
der er hyppigere i ét delkorpus end i resten. Et enkelt mål er **log-ratio**, som er log2 af forholdet
mellem ordets relative frekvens i området og i resten af korpusset.

$$\text{log-ratio} = \log_2 \frac{(n + 0{,}5) / N_{\text{område}}}{(n_{\text{rest}} + 0{,}5) / N_{\text{rest}}}$$

En log-ratio på 1 betyder, at ordet er dobbelt så hyppigt i området som i resten. Værdien 0,5 indgår i ligningen, så vi ikke dividerer med 0, når et ord slet ikke står i resten.

**Opgave:**

Brug `indhold` (uden stopord).

1. Tæl, hvor mange gange hvert ord står i hvert område: en tabel med kolonnerne `omraade`,
   `ord` og `n`.
2. Tilføj `N_omr` (antal ord i området i alt) og `n_alle` (ordets antal i hele korpusset).
3. Beregn `n_rest`, `N_rest` og `log_ratio`.
4. Behold ord med `n >= 50`, og find de 8 ord med højest log-ratio i hvert område.

**Hjælp:**

- Slides, *TF-IDF*: `dtm.div(dtm.sum(axis=1), axis=0)` giver relative frekvenser, og `np.log()`. Her skal I bruge `np.log2()`.
- `.groupby(...)["n"].transform("sum")` giver summen for gruppen på hver række, så den kan bruges i en beregning række for række.
- spaCy-notebooken, trin 5.6: `.groupby(level=0).head(10)`. Her grupperer I på `"omraade"`.

In [ ]:
# Din kode her
# Antal gange hvert ord står i hvert område (kolonnerne omraade, ord og n)
n = indhold.groupby([____, ____]).size().rename("n").reset_index()

print(n.head())

In [ ]:
# Summer, der lægges på hver række med .transform()
n["N_omr"] = n.groupby("omraade")["n"].transform("sum")   # antal ord i området i alt
n["n_alle"] = ____                                          # ordets antal i hele korpusset

# "Resten" er alt minus området
n["n_rest"] = ____
n["N_rest"] = n["n"].sum() - n["N_omr"]

In [ ]:
# log-ratio (skriv ligningen ovenfor som kode)
n["log_ratio"] = np.log2(((n["n"] + 0.5) / n["N_omr"]) /
                         ((____ + 0.5) / ____))

In [ ]:
# Kun ord med n >= 50, og de 8 med højest log-ratio i hvert område
noegleord = (____
             .sort_values("log_ratio", ascending=False)
             .groupby("omraade")
             .head(____))

print(noegleord.groupby("omraade")["ord"].agg(list))

**Kig efter i viewer:** Åbn `n` (eller hvad i kalder jeres variabel), og sortér efter `log_ratio`. Hvad står øverst, hvis I ikke har fjernet de sjældne ord?

**Til diskussion:**

- Sammenlign med A4. Er det nu lettere at genkende områderne?
- Nogle nøgleord er **personnavne**. Hvorfor? Hvad betyder det, hvis vi i en klyngeanalyse vil
  finde områderne ud fra, *hvad der tales om*?
- Hvorfor kræver vi en nedre grænse for `n`?

*Jeres svar:*

---

## B2. Et tema, debat for debat og med usikkerhed

I slides målte vi temaer ved at lægge alle ord fra et år sammen (*Over tid*). Her måler vi i
stedet **per debat** og spørger, hvor sikre tallene er. Vi bruger udlændingeordbogen fra slides.

**Opgave:**

```python
udl = ["udlændinge", "flygtninge", "asyl", "asylansøgere",
       "indvandrere", "integration", "opholdstilladelse", "udvisning"]
```

1. Lav kolonnen `udl` i `lang`: `True` for hvert ord, der står i `udl`.
2. Beregn for hver debat, hvor mange udlændingeord der er per 1.000 ord. Gem det som kolonnen
   `udl` i `korpus`.
3. Hvor stor en andel af debatterne har værdien 0? Lav en oversigt med gennemsnit, median og
   andel med 0 for hvert politikområde.
4. Beregn for hvert år både det samlede mål fra slides (alle ord i året lagt sammen) og
   gennemsnittet af debatterne. Sammenlign dem.
5. Lav et 95 %-bootstrap-interval for gennemsnittet i hvert år: træk debatter med tilbagelægning
   inden for hvert år 500 gange, og tag 2,5 %- og 97,5 %-fraktilen.
6. Tegn gennemsnittet og intervallet over tid.

**Hjælp:**

- Slides, *Mål: per 1.000 ord*: `lang["ord"].isin(kriseord)` og `lang.groupby("aar")[...].mean() * 1000`.
- Slides, *Kør den på korpusset*: `.groupby(level=0)` giver én værdi per debat.
- Preprocess-notebooken, trin 5.2: `.agg(...)` med navngivne kolonner.
- Bootstrap: `korpus.groupby("aar").sample(frac=1, replace=True, random_state=i)` trækker én ny stikprøve. Gentag med en list comprehension, og saml med `pd.concat(..., axis=1)` og `.quantile(0.025, axis=1)`.
- Slides, *Rammer data forudsigelsen?*: plotnine. Intervallet tegnes med `geom_ribbon(aes(ymin=..., ymax=...))`.

In [ ]:
# Din kode her
udl = ["udlændinge", "flygtninge", "asyl", "asylansøgere",
       "indvandrere", "integration", "opholdstilladelse", "udvisning"]

# True for hvert ord i lang, der står i udl
lang["udl"] = ____

In [ ]:
# Udlændingeord per 1.000 ord i hver debat. Gennemsnittet af True/False per debat er andelen af udlændingeord.
korpus["udl"] = lang["udl"].groupby(level=0).____() * 1000

print(korpus["udl"].describe().round(2))
print()

# Andel debatter med 0, i alt og per område
print("Andel debatter med 0:", round((____).mean(), 3))
print()
print(korpus.groupby("omraade")["udl"].agg(
    gennemsnit="mean",
    median="____",
    andel_nul=lambda s: ____).round(2))   # samme som linjen ovenfor, men på s

In [ ]:
# 4. To mål per år
pr_aar = pd.DataFrame({
    "samlet": lang.groupby("aar")["udl"].mean() * 1000,   # alle ord i året lagt sammen, som i slides
    "gns_debat": ____,                                      # gennemsnittet af debatternes udl
}).round(2)

print(pr_aar)

In [ ]:
# Bootstrap:
#    Ét træk: træk debatterne med tilbagelægning inden for hvert år, og tag gennemsnittet per år.
#    Listen [... for i in range(...)] gentager trækket, med et nyt random_state hver gang.
#    pd.concat(..., axis=1) sætter trækkene ved siden af hinanden (én række per år, én kolonne per træk.)
#    Forslag: prøv først med range(5), så det går hurtigt.
traek = pd.concat([korpus.groupby("aar")
                         .sample(frac=1, replace=True, random_state=____)
                         .groupby("aar")["udl"].mean()
                   for i in range(____)], axis=1)

# Fraktilerne tages på tværs af trækkene, dvs. hen ad hver række (axis=1)
usikkerhed = pd.DataFrame({
    "gns": korpus.groupby("aar")["udl"].mean(),
    "nedre": traek.quantile(____, axis=1),
    "oevre": traek.quantile(____, axis=1),
}).round(2)

print(usikkerhed)

In [ ]:
# Gennemsnit og interval over tid
(ggplot(usikkerhed.reset_index(), aes(x="aar", y="gns"))
 + geom_ribbon(aes(ymin="____", ymax="____"), alpha=0.2)   # intervallet som et bånd
 + geom_line()
 + geom_point()
 + labs(x="År", y="Udlændingeord per 1.000 ord",
        title="Udlændingetemaet per debat, med 95 %-bootstrap-interval")
 + theme_classic())

**Kig efter i viewer:** Sortér `korpus` efter `udl`. Hvilke debatter scorer højest? Hvilket område kommer de fra?

**Til diskussion:**

- Hvordan ser fordelingen ud? Hvad betyder det for gennemsnittet som mål?
- Hvorfor er det samlede mål og gennemsnittet af debatterne ikke det samme? Hvilke debatter
  vejer mest i det samlede mål? (Tænk på A2.)
- Kan I sige, at udlændingetemaet fyldte mere i 2015-2017 end i årene før? Hvor brede er
  intervallerne?

*Jeres svar:*